# Day 03 — Problemin Bilgisayar Mühendisliği Açısından Tanımlanması
## Problem, Girdi, Beklenen Çıktı, Başarı Ölçütü ve Basit Başlangıç Yöntemi (Baseline)

> **Aşama:** Faz 1 — Problem, Veri ve Geliştirme Temelleri (Day 01–08)
> **Resmi Staj Defteri Konusu:** Problemin Bilgisayar Mühendisliği Açısından Tanımlanması (Yaprak 5 & 6)

### 1. Problem
Endüstriyel yapay zekâ projelerinde sıkça yapılan hata, neyin çözülmek istendiği, girdinin ve beklenen çıktının sınırları tam çizilmeden doğrudan karmaşık modellere yönelmektir. Örneğin 'halılar benziyor mu?' sorusu bilgisayar mühendisliği açısından tanımsızdır; renk benzerliği mi, motif benzerliği mi, dokuma sıklığı mı kastedildiği netleştirilmelidir.

### 2. Why the Problem Matters
Bir algoritmanın başarılı kabul edilebilmesi için önce en basit başlangıç yöntemiyle (heuristic/baseline) elde edilen skorun bilinmesi gerekir. Basit bir kural tabanlı yöntem %85 doğruluk veriyorsa, karmaşık bir yapay zekâ modelinin buna kıyasla getirdiği kazanç hesaplanmalıdır.

### 3. Engineering Concepts
- **Girdi / Çıktı Sözleşmesi (Input/Output Contract)**: Sisteme girecek veri tipleri ve üretilecek yanıt biçimi.
- **Başarı Ölçütü (Evaluation Metric)**: Doğruluk (Accuracy), Hassasiyet (Precision), Top-k sıralama başarımı.
- **En Basit Başlangıç Yöntemi (Baseline)**: Karşılaştırma referansı oluşturan basit kural veya sezgisel yöntem.

In [ ]:
# 4. Library / API Investigation
from day03.mini_project.src.problem_spec import ProblemSpecification, BaselineEvaluator

spec = ProblemSpecification(
    problem_id="PROB-SIM-01",
    name="Carpet Visual Similarity",
    input_contract={"query_image": "ndarray", "catalog": "List[Image]"},
    output_contract={"top_5_matches": "List[Tuple[str, float]]"},
    target_metric="Top-1 Accuracy",
    baseline_threshold=0.60,
    latency_sla_ms=25.0
)
print("Problem Sözleşmesi:", spec.model_dump_json(indent=2))

In [ ]:
# 5. Minimal Implementation
evaluator = BaselineEvaluator(spec)
ground_truth = ["A", "B", "A", "C", "B", "A", "B", "C"]
baseline_preds = ["A", "A", "A", "A", "A", "A", "A", "A"]  # Sabit çoğunluk sınıfı baseline'ı
candidate_preds = ["A", "B", "A", "B", "B", "A", "B", "C"] # Geliştirilen aday algoritma

comparison = evaluator.evaluate(ground_truth, baseline_preds, candidate_preds, baseline_lat_ms=0.01, candidate_lat_ms=1.12)
print(comparison.summary)

In [ ]:
# 6. Experiment: Baseline vs Candidate Performance
print(f"Baseline Doğruluk: {comparison.baseline_metric:.2%}")
print(f"Aday Model Doğruluk: {comparison.candidate_metric:.2%}")
print(f"Göreceli Kazanç: +{comparison.relative_improvement_pct:.1f}%")
print(f"Aday Model Eşiği Aştı mı? -> {comparison.is_candidate_superior}")

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

models = ["Basit Baseline (Sabit Tahmin)", "Aday Yöntem"]
accuracies = [comparison.baseline_metric * 100, comparison.candidate_metric * 100]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies, color=["#999999", "#2ca02c"], width=0.5)
plt.axhline(spec.baseline_threshold * 100, color="red", linestyle="--", label=f"Hedef Eşik (%{spec.baseline_threshold*100:.0f})")
plt.ylabel("Doğruluk (%) - Accuracy")
plt.title("Problemin Başarı Ölçütüne Göre Değerlendirilmesi")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert comparison.candidate_metric >= spec.baseline_threshold
assert comparison.is_candidate_superior is True
print("Aday yöntem hem baseline'ı geçmiş hem de hedef eşiği aşmıştır.")

In [ ]:
# 9. Failure Cases: Empty or mismatched predictions
try:
    evaluator.evaluate([], [], [], 0, 0)
except ValueError as e:
    print("Beklenen hata yakalandı:", e)

### 10. Conclusions
Bir problemin mühendislik sınırları girdi, çıktı ve metrik bazında netleştirilmiş; basit başlangıç yönteminin (baseline) ölçülmesiyle yeni geliştirilecek algoritmalar için somut kıyaslama temeli oluşturulmuştur.